# LLIE Colab L4 Pipeline

这份 notebook 面向 Colab `L4 GPU`，用于完成低光照增强论文的第一轮实验复现。

当前主线：

- `URetinex-Net` on `LOL-v1`
- `Diff-Retinex` on `LOL-v1`
- `Reti-Diff` on `LOLv2-Real` and `LOLv2-Synthetic`
- 自动记录状态、日志、指标与汇总表


## 初始化

默认优先使用从 GitHub 克隆下来的仓库代码；Google Drive 只存数据集、权重和运行结果。

如果远端运行环境里还没有 `/content/sspir`，下面的代码会自动克隆仓库。


In [14]:
from google.colab import drive
from pathlib import Path
import json
import subprocess
import shutil

drive_root = Path('/content/drive')
if not drive_root.exists() or not any(drive_root.iterdir()):
    drive.mount('/content/drive')
else:
    print('Google Drive is already mounted at /content/drive')

REPO_URL = 'https://github.com/yimeng-tong/sspir.git'
REPO_ROOT = Path('/content/sspir')
repo_notebook_root = REPO_ROOT / 'experiments' / 'colab_l4'

if not (repo_notebook_root / 'config' / 'run_config.example.json').exists():
    if REPO_ROOT.exists() and not (REPO_ROOT / '.git').exists():
        raise RuntimeError(
            'Path /content/sspir exists but is not a git repo. Remove it or rename it, then rerun.'
        )
    if not REPO_ROOT.exists():
        print(f'Cloning repository to {REPO_ROOT} ...')
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)

candidate_roots = [
    repo_notebook_root,
    Path('/content/drive/MyDrive/thesis/experiments/colab_l4'),
    Path.cwd(),
    Path.cwd() / 'experiments' / 'colab_l4',
]

NOTEBOOK_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'config' / 'run_config.example.json').exists():
        NOTEBOOK_ROOT = candidate
        break

if NOTEBOOK_ROOT is None:
    raise FileNotFoundError(
        'Could not locate experiments/colab_l4. If you cloned the repo, use '
        '/content/sspir/experiments/colab_l4. If you uploaded code to Drive, use '
        '/content/drive/MyDrive/thesis/experiments/colab_l4.'
    )

CONFIG_PATH = NOTEBOOK_ROOT / 'config' / 'run_config.json'
EXAMPLE_CONFIG = NOTEBOOK_ROOT / 'config' / 'run_config.example.json'

if not CONFIG_PATH.exists():
    shutil.copyfile(EXAMPLE_CONFIG, CONFIG_PATH)
    print(f'Created {CONFIG_PATH}. Edit it before continuing.')

with CONFIG_PATH.open('r', encoding='utf-8') as f:
    cfg = json.load(f)

print('Notebook root:', NOTEBOOK_ROOT)
print(json.dumps(cfg, ensure_ascii=False, indent=2))


Google Drive is already mounted at /content/drive
Notebook root: /content/sspir/experiments/colab_l4
{
  "note": "Copy this file to run_config.json beside the notebook code root and replace every placeholder path before running the notebook. Keep datasets, weights and runs on Google Drive; code can stay in the cloned GitHub repo.",
  "drive_root": "/content/drive/MyDrive/thesis_llie_l4",
  "repos_root": "/content/thesis_llie_repos",
  "output_root": "/content/drive/MyDrive/thesis_llie_l4/runs",
  "datasets": {
    "lol_v1": {
      "lq_dir": "/content/drive/MyDrive/thesis_llie_l4/datasets/LOL/eval15/low",
      "gt_dir": "/content/drive/MyDrive/thesis_llie_l4/datasets/LOL/eval15/high"
    },
    "lolv2_real": {
      "lq_dir": "/content/drive/MyDrive/thesis_llie_l4/datasets/LOLv2/Real_captured/Test/Low",
      "gt_dir": "/content/drive/MyDrive/thesis_llie_l4/datasets/LOLv2/Real_captured/Test/Normal"
    },
    "lolv2_syn": {
      "lq_dir": "/content/drive/MyDrive/thesis_llie_l4/datase

In [15]:
!nvidia-smi

with CONFIG_PATH.open('r', encoding='utf-8') as f:
    cfg = json.load(f)

REPOS_ROOT = Path(cfg['repos_root'])
OUTPUT_ROOT = Path(cfg['output_root'])
REPOS_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('Repos root:', REPOS_ROOT)
print('Output root:', OUTPUT_ROOT)


Sun Mar 29 18:51:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   38C    P8             16W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [16]:
!pip -q install pyyaml lpips scikit-image tensorboardX timm einops thop pyiqa openai-clip ftfy addict natsort lmdb facexlib


In [17]:
import subprocess

repos = {
    'URetinex-Net': 'https://github.com/SZU-AdvTech-2023/262-URetinex-Net-Retinex-based-Deep-Unfolding-Network-for-Low-light-Image-Enhancement.git',
    'Diff-Retinex': 'https://github.com/XunpengYi/Diff-Retinex.git',
    'Reti-Diff': 'https://github.com/ChunmingHe/Reti-Diff.git',
}

for name, url in repos.items():
    target = REPOS_ROOT / name
    if target.exists():
        print(f'Skip clone, already exists: {target}')
    else:
        subprocess.run(['git', 'clone', '--depth', '1', url, str(target)], check=True)

subprocess.run(['pip', 'install', '-q', '-r', str(REPOS_ROOT / 'Reti-Diff' / 'requirements.txt')], check=True)
subprocess.run(['pip', 'install', '-q', '-e', str(REPOS_ROOT / 'Reti-Diff')], check=True)

print('Repositories are ready.')


Skip clone, already exists: /content/thesis_llie_repos/URetinex-Net
Skip clone, already exists: /content/thesis_llie_repos/Diff-Retinex
Skip clone, already exists: /content/thesis_llie_repos/Reti-Diff
Repositories are ready.


## 运行前检查

如果你修改了 `run_config.json`，下面的单元会重新读取它，所以不需要重启 notebook。


In [18]:
import subprocess
import json

with CONFIG_PATH.open('r', encoding='utf-8') as f:
    cfg = json.load(f)

METHOD_DATASETS = {
    'uretinex': ['lol_v1'],
    'diff-retinex': ['lol_v1'],
    'reti-diff': ['lolv2_real', 'lolv2_syn'],
}

SCRIPT = NOTEBOOK_ROOT / 'scripts' / 'run_baseline.py'

for method, dataset_keys in METHOD_DATASETS.items():
    if not cfg['run_switches'].get(method.replace('-', '_'), True):
        print(f'Skip disabled method: {method}')
        continue
    for dataset_key in dataset_keys:
        ds = cfg['datasets'][dataset_key]
        cmd = [
            'python', str(SCRIPT),
            '--method', method,
            '--output-root', str(OUTPUT_ROOT),
            '--dataset-name', dataset_key,
            '--lq-dir', ds['lq_dir'],
            '--gt-dir', ds['gt_dir'],
        ]

        if method == 'uretinex':
            cmd.extend(['--repo-root', str(REPOS_ROOT / 'URetinex-Net')])
        elif method == 'diff-retinex':
            cmd.extend([
                '--repo-root', str(REPOS_ROOT / 'Diff-Retinex'),
                '--diff-tdn-weight', cfg['weights']['diff_retinex']['tdn'],
                '--diff-rda-weight', cfg['weights']['diff_retinex']['rda'],
                '--diff-ida-weight', cfg['weights']['diff_retinex']['ida'],
            ])
        elif method == 'reti-diff':
            template_name = 'test_LLIE_real.yml' if dataset_key == 'lolv2_real' else 'test_LLIE_syn.yml'
            weight_name = 'llie_real' if dataset_key == 'lolv2_real' else 'llie_syn'
            cmd.extend([
                '--repo-root', str(REPOS_ROOT / 'Reti-Diff'),
                '--reti-template', str(REPOS_ROOT / 'Reti-Diff' / 'options' / template_name),
                '--reti-weight', cfg['weights']['reti_diff'][weight_name],
                '--reti-decom-weight', cfg['weights']['reti_diff']['retinex_decom'],
            ])

        print('Running:', ' '.join(cmd))
        subprocess.run(cmd, check=True)


Running: python /content/sspir/experiments/colab_l4/scripts/run_baseline.py --method uretinex --output-root /content/drive/MyDrive/thesis_llie_l4/runs --dataset-name lol_v1 --lq-dir /content/drive/MyDrive/thesis_llie_l4/datasets/LOL/eval15/low --gt-dir /content/drive/MyDrive/thesis_llie_l4/datasets/LOL/eval15/high --repo-root /content/thesis_llie_repos/URetinex-Net
Skip disabled method: diff-retinex
Skip disabled method: reti-diff


In [19]:
import json
import subprocess
from pathlib import Path

with CONFIG_PATH.open('r', encoding='utf-8') as f:
    cfg = json.load(f)

enabled_methods = {
    method.replace('_', '-'): enabled
    for method, enabled in cfg['run_switches'].items()
    if method != 'compute_metrics' and enabled
}

metrics_script = NOTEBOOK_ROOT / 'scripts' / 'compute_metrics.py'
meta_files = sorted(OUTPUT_ROOT.rglob('run_metadata.json'))

for meta_file in meta_files:
    meta = json.loads(meta_file.read_text(encoding='utf-8'))
    method = meta['method']
    if method not in enabled_methods:
        print(f'Skip disabled method metadata: {meta_file}')
        continue

    status_path = Path(meta['status_path']) if meta.get('status_path') else None
    if status_path and status_path.exists():
        run_status = json.loads(status_path.read_text(encoding='utf-8'))
        if run_status.get('state') != 'completed':
            print(f'Skip incomplete run: {meta_file}')
            continue

    pred_dir = Path(meta['pred_dir'])
    if not pred_dir.exists():
        print(f'Skip missing prediction dir: {pred_dir}')
        continue

    pred_images = [
        p for p in pred_dir.rglob('*')
        if p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp'}
    ]
    if not pred_images:
        print(f'Skip empty prediction dir: {pred_dir}')
        continue

    run_dir = meta_file.parent
    metrics_dir = run_dir / 'metrics'
    cmd = [
        'python', str(metrics_script),
        '--method', method,
        '--dataset-name', meta['dataset_name'],
        '--pred-dir', str(pred_dir),
        '--gt-dir', meta['gt_dir'],
        '--output-dir', str(metrics_dir),
    ]
    if meta.get('status_path'):
        cmd.extend(['--status-path', meta['status_path']])
    if method == 'uretinex':
        cmd.extend(['--strip-suffix', '_URetinexNet'])
    cmd.extend(['--skip-empty', '--skip-no-match'])
    if cfg['run_switches'].get('compute_metrics', True):
        print('Computing metrics:', ' '.join(cmd))
        subprocess.run(cmd, check=True)


Skip disabled method metadata: /content/drive/MyDrive/thesis_llie_l4/runs/diff-retinex/lol_v1/20260329_184008/run_metadata.json
Skip incomplete run: /content/drive/MyDrive/thesis_llie_l4/runs/uretinex/lol_v1/20260329_183310/run_metadata.json
Skip incomplete run: /content/drive/MyDrive/thesis_llie_l4/runs/uretinex/lol_v1/20260329_183907/run_metadata.json
Computing metrics: python /content/sspir/experiments/colab_l4/scripts/compute_metrics.py --method uretinex --dataset-name lol_v1 --pred-dir /content/drive/MyDrive/thesis_llie_l4/runs/uretinex/lol_v1/20260329_184004/predictions --gt-dir /content/drive/MyDrive/thesis_llie_l4/datasets/LOL/eval15/high --output-dir /content/drive/MyDrive/thesis_llie_l4/runs/uretinex/lol_v1/20260329_184004/metrics --status-path /content/drive/MyDrive/thesis_llie_l4/runs/uretinex/lol_v1/20260329_184004/status.json --strip-suffix _URetinexNet


CalledProcessError: Command '['python', '/content/sspir/experiments/colab_l4/scripts/compute_metrics.py', '--method', 'uretinex', '--dataset-name', 'lol_v1', '--pred-dir', '/content/drive/MyDrive/thesis_llie_l4/runs/uretinex/lol_v1/20260329_184004/predictions', '--gt-dir', '/content/drive/MyDrive/thesis_llie_l4/datasets/LOL/eval15/high', '--output-dir', '/content/drive/MyDrive/thesis_llie_l4/runs/uretinex/lol_v1/20260329_184004/metrics', '--status-path', '/content/drive/MyDrive/thesis_llie_l4/runs/uretinex/lol_v1/20260329_184004/status.json', '--strip-suffix', '_URetinexNet']' returned non-zero exit status 1.

In [ ]:
import json
import subprocess

with CONFIG_PATH.open('r', encoding='utf-8') as f:
    cfg = json.load(f)

enabled_methods = [
    method.replace('_', '-')
    for method, enabled in cfg['run_switches'].items()
    if method != 'compute_metrics' and enabled
]

aggregate_script = NOTEBOOK_ROOT / 'scripts' / 'aggregate_results.py'
summary_root = OUTPUT_ROOT
aggregate_root = OUTPUT_ROOT / 'aggregated'
cmd = [
    'python', str(aggregate_script),
    '--input-root', str(summary_root),
    '--output-root', str(aggregate_root),
    '--only-completed',
]
for method in enabled_methods:
    cmd.extend(['--allowed-method', method])

subprocess.run(cmd, check=True)
print((aggregate_root / 'aggregated_metrics.md').read_text(encoding='utf-8'))


## 后续建议

1. 第一轮先把这三条基线结果跑完整。
2. 记录 `status.json`、`run.log`、`summary.json` 到 Drive。
3. 再回到本地论文工作区，把均值指标与典型可视化填入第四章。
4. 如果后续还要扩展到 `SSP-IR`，建议单独开一个 notebook，不要和当前三条基线混在一起。
